# Click prompts collector — whip videos

Runs on **olab-1's local Jupyter** (the one you already have at `http://127.0.0.1:8888/`). 
Frames stay on bigpurple. For each video we:

1. Ask bigpurple how many frames the video has (one ssh ls).
2. Rsync **just the 3 prompt frames** we need to click on (~5 MB per video).
3. You click. Left = positive, right = negative. Number keys 1/2/3 switch object id.
4. Save `prompts/<video>.json` locally and push it to bigpurple.

When you're done, on bigpurple run `sbatch --array=0-$((N-1))%8 run_inference_array.sh` to fan the inference out.

In [ ]:
%matplotlib widget
from pathlib import Path
import json, subprocess, os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

BP_HOST      = 'bigpurple'           # ssh alias
BP_FRAMES    = '/gpfs/data/oermannlab/private_data/whip/frames_attempt2'
BP_REPO      = '/gpfs/data/oermannlab/users/schula12/Surgical-SAM-2'

CACHE_DIR    = Path('test_data/_prompt_frames')   # 3 frames per video, ~5 MB each
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS_DIR  = Path('prompts')
PROMPTS_DIR.mkdir(exist_ok=True)

N_PROMPT_FRAMES = 3

OBJ_COLORS = ['#00ff00', '#ff8800', '#00bfff', '#ff00ff', '#ffff00',
              '#ff0000', '#00ffff', '#ffffff']

def ssh(cmd):
    """Run a command on bigpurple, return stdout (stripped). Loud on failure."""
    r = subprocess.run(['ssh', BP_HOST, cmd], capture_output=True, text=True)
    if r.returncode != 0:
        print('ssh stderr:', r.stderr)
        r.check_returncode()
    out = r.stdout.strip()
    # bigpurple's login shell adds a banner; strip it
    lines = [l for l in out.splitlines() if 'Loading' not in l and 'requirement' not in l]
    return '\n'.join(lines)

def list_videos():
    """Return [(video_name, frame_count), ...] for videos with >=3 frames."""
    raw = ssh(f'for d in {BP_FRAMES}/*/; do '
              f'  n=$(ls $d 2>/dev/null | wc -l); '
              f'  printf "%s %d\\n" "$(basename $d)" $n; '
              f'done')
    out = []
    for line in raw.splitlines():
        parts = line.strip().rsplit(' ', 1)
        if len(parts) != 2: continue
        name, n_s = parts
        if name.startswith('480full'): continue
        try:
            n = int(n_s)
        except ValueError:
            continue
        if n >= 3:
            out.append((name, n))
    return out

videos = list_videos()
print(f'{len(videos)} videos with >=3 frames on bigpurple')
for v, n in videos[:5]:
    print(f'  {v}: {n} frames')
if len(videos) > 5:
    print(f'  ... and {len(videos)-5} more')

In [ ]:
def prompt_frame_indices(n_total, n_prompt=N_PROMPT_FRAMES):
    return [int(i) for i in np.linspace(0, n_total-1, n_prompt, dtype=int)]

def fetch_prompt_frames(video_name, n_total):
    """Rsync exactly the 3 prompt frames for this video into a local cache.
    Returns the loader indices (0..n_total-1) chosen and the local file paths."""
    idxs = prompt_frame_indices(n_total)
    # Need to know the SORTED filenames on bigpurple so we can pick the right ones.
    # Use ls -- naming is 'frame_0000000000.png' style; sort lexically == numerically
    # when the digit count is uniform (which it is for the whip set).
    raw = ssh(f'cd {BP_FRAMES}/{video_name} && ls | sort')
    all_files = raw.splitlines()
    assert len(all_files) == n_total, (
        f'{video_name}: expected {n_total} files, ssh ls returned {len(all_files)}'
    )
    chosen = [(i, all_files[i]) for i in idxs]

    local_dir = CACHE_DIR / video_name
    local_dir.mkdir(parents=True, exist_ok=True)
    local_paths = []
    for i, fname in chosen:
        target = local_dir / f'idx{i:07d}__{fname}'
        if not target.exists():
            subprocess.run(
                ['rsync', '-az',
                 f'{BP_HOST}:{BP_FRAMES}/{video_name}/{fname}', str(target)],
                check=True, capture_output=True)
        local_paths.append((i, target))
    return local_paths

# Quick test of the fetcher on the first un-done video
def first_undone():
    for v, n in videos:
        if not (PROMPTS_DIR / f'{v}.json').exists():
            return v, n
    return None, None

_v, _n = first_undone()
if _v:
    print(f'Next un-done: {_v} ({_n} frames). Run the click cell below.')
else:
    print('Every video already has a prompts JSON. Nothing to do.')

In [ ]:
def collect_clicks_for_frame(image_path, title):
    """Open one figure, return {obj_id: {'positive':[[x,y]...], 'negative':[...]}}.
    User closes the figure when finished with this frame."""
    img = Image.open(image_path)
    fig, ax = plt.subplots(figsize=(13, 7.5))
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    fig.tight_layout()

    state = {'cur': 1}
    result = {}
    info_text = ax.text(0.01, 0.99, '', transform=ax.transAxes,
                        ha='left', va='top', fontsize=11, color='black')

    def refresh():
        info_text.set_text(f'obj {state["cur"]}  '
                           f'(Left=positive, Right=negative, press 1/2/3 to switch, close window when done)')
        info_text.set_bbox(dict(facecolor=OBJ_COLORS[(state['cur']-1) % len(OBJ_COLORS)],
                                alpha=0.85, edgecolor='none'))

    def on_click(event):
        if event.inaxes != ax or event.xdata is None: return
        oid = state['cur']
        result.setdefault(oid, {'positive': [], 'negative': []})
        color = OBJ_COLORS[(oid-1) % len(OBJ_COLORS)]
        if event.button == 1:
            result[oid]['positive'].append([round(event.xdata, 1), round(event.ydata, 1)])
            ax.plot(event.xdata, event.ydata, marker='+', color=color, ms=22, mew=3)
        elif event.button == 3:
            result[oid]['negative'].append([round(event.xdata, 1), round(event.ydata, 1)])
            ax.plot(event.xdata, event.ydata, marker='x', color=color, ms=22, mew=3)
        fig.canvas.draw_idle()

    def on_key(event):
        if event.key and event.key.isdigit() and event.key != '0':
            state['cur'] = int(event.key)
            refresh()
            fig.canvas.draw_idle()

    refresh()
    fig.canvas.mpl_connect('button_press_event', on_click)
    fig.canvas.mpl_connect('key_press_event', on_key)
    plt.show()
    return result

def push_prompts_to_bigpurple(json_path):
    subprocess.run(
        ['rsync', '-az', str(json_path),
         f'{BP_HOST}:{BP_REPO}/prompts/'],
        check=True)

def collect_for_video(video_name, n_total, push=True):
    print(f'\n=== {video_name} ({n_total} frames) ===')
    chosen = fetch_prompt_frames(video_name, n_total)
    objects_by_frame = {}
    first_img = Image.open(chosen[0][1])
    W, H = first_img.size
    for k, (loader_idx, local_path) in enumerate(chosen, 1):
        title = (f'{video_name} — prompt {k}/{len(chosen)} '
                 f'(loader idx {loader_idx}; close window to advance)')
        clicks = collect_clicks_for_frame(local_path, title)
        if clicks:
            objects_by_frame[int(loader_idx)] = [
                {'obj_id': oid, 'positive': v['positive'], 'negative': v['negative']}
                for oid, v in sorted(clicks.items())
            ]
        else:
            print(f'  (no clicks at loader idx {loader_idx})')
    out = {
        'video': video_name,
        'resolution': [W, H],
        'n_frames': n_total,
        'prompt_frames': [i for i, _ in chosen],
        'objects_by_frame': {str(k): v for k, v in objects_by_frame.items()},
    }
    out_path = PROMPTS_DIR / f'{video_name}.json'
    with open(out_path, 'w') as fp:
        json.dump(out, fp, indent=2)
    print(f'  saved local: {out_path}')
    if push:
        push_prompts_to_bigpurple(out_path)
        print(f'  pushed to:  {BP_HOST}:{BP_REPO}/prompts/{out_path.name}')
    return out

print('Helpers loaded.')

## Click one video

In [ ]:
VIDEO = 'DG_whip_16598313'   # <-- pick from the list printed above
n_total = dict(videos)[VIDEO]
out = collect_for_video(VIDEO, n_total)
print(json.dumps(out, indent=2)[:600])

## Walk every un-done video

In [ ]:
LIMIT = 5   # set to None for all of them
done = 0
for vname, n_total in videos:
    if (PROMPTS_DIR / f'{vname}.json').exists():
        continue
    collect_for_video(vname, n_total)
    done += 1
    if LIMIT is not None and done >= LIMIT:
        break
n_done = sum(1 for v,_ in videos if (PROMPTS_DIR/(v+'.json')).exists())
print(f'\nThis session: {done} videos. Total done: {n_done} / {len(videos)}.')